# 00 · End-to-end: game video → tracking data, touch-by-touch events, 3D play

Runs the whole CourtVision pipeline on one video in Colab:

1. detect + track players, referees, ball and hoop
2. court keypoints → homography → camera (per frame)
3. clean the ball track (select, de-outlier, interpolate)
4. assign teams by jersey colour
5. ball possession with hysteresis
6. passes, interceptions, turnovers, shots (made/missed, 2PT/3PT), rebounds
7. court positions in metres, speed and distance
8. 3D ball height from physics, player body poses (optional), animated 3D scene

Weights come from notebooks 01 and 02, saved in `MyDrive/courtvision/models/`.

In [ ]:
# Runtime > Change runtime type > GPU before running.
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
!git clone -q {REPO_URL} /content/courtvision || (cd /content/courtvision && git pull -q)
%cd /content/courtvision
!pip -q install -e ".[models]"

## Configure

In [ ]:
from courtvision import Config, Pipeline

cfg = Config.load("configs/default.yaml")
cfg.models.detector = f"{DRIVE_ROOT}/models/player_detector.pt"
cfg.models.court_keypoints = f"{DRIVE_ROOT}/models/court_keypoints.pt"
if os.path.exists(f"{DRIVE_ROOT}/models/ball_detector.pt"):
    cfg.models.ball_detector = f"{DRIVE_ROOT}/models/ball_detector.pt"
cfg.teams.team_names = ["Lakers", "Spurs"]   # display names; team 0/1 is decided by jersey colour
cfg.output.dir = f"{DRIVE_ROOT}/output"
cfg.output.cache_dir = f"{DRIVE_ROOT}/stubs"  # detections are cached; re-runs skip the GPU pass

# Player body poses from video: a YOLO pose pass, lifted to 3D skeletons in scene.json.
# Adds GPU time plus ~10 ms per player per frame to lift, so turn it on for clips, not full games.
cfg.pose.enabled = True

# Rigged character for the 3D viewer: any Mixamo-rigged .glb (see the next cell). Without one,
# the viewer draws a built-in mannequin.
CHARACTER = f"{DRIVE_ROOT}/models/player.glb"
if os.path.exists(CHARACTER):
    cfg.output.character_glb = CHARACTER

### Optional: a rigged player model

The 3D viewer drives any humanoid with Mixamo bone names (`mixamorig:Hips`, `LeftUpLeg`, `LeftArm`, ...).

* **Your own:** pick a character on [mixamo.com](https://www.mixamo.com), download it as FBX in T-pose, convert it to `.glb` in Blender (File > Import FBX, then File > Export glTF 2.0), and save it as `MyDrive/courtvision/models/player.glb`.
* **Quick test:** the next cell downloads the X Bot sample model from the three.js examples. Check its licence before using it outside personal testing.

In [ ]:
# Quick test model (skip if you already have player.glb)
if not os.path.exists(CHARACTER):
    !wget -q -O "{CHARACTER}" https://raw.githubusercontent.com/mrdoob/three.js/r147/examples/models/gltf/Xbot.glb
    cfg.output.character_glb = CHARACTER

## Pick a video (upload, or put it in `MyDrive/courtvision/input_videos/`)

In [ ]:
import glob
videos = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))
if not videos:
    from google.colab import files
    up = files.upload()
    videos = list(up)
VIDEO = videos[0]
VIDEO

## Run

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
result = Pipeline(cfg).run(VIDEO)
OUT = os.path.join(cfg.output.dir, os.path.splitext(os.path.basename(VIDEO))[0])
print(f"{len(result.players)} frames · {len(result.team_of)} player tracks · "
      f"{len(result.possessions)} touches · {len(result.events)} events")

## Touch-by-touch data

In [ ]:
import pandas as pd
events = pd.read_csv(f"{OUT}/events.csv")
touches = pd.read_csv(f"{OUT}/touches.csv")
display(events.head(30))
display(touches.head(30))

In [ ]:
display(pd.read_csv(f"{OUT}/box_score.csv"))

## Tracking data (metres, every player and the ball, every frame)

In [ ]:
tracking = pd.read_csv(f"{OUT}/tracking.csv")
display(tracking.head())
tracking[tracking.object == "player"].groupby("id")[["distance_m", "speed_kmh"]].max().sort_values("distance_m", ascending=False)

## 3D play reconstruction

Animated players: speed drives the gait, defenders face the ball, shots and passes trigger jumps and throws, and measured skeletons from the pose stage replace the synthesized limbs. Use **Free orbit** to move the camera; **Open scene.json** and **Load character** also work on the saved file outside Colab.

In [ ]:
from IPython.display import HTML
HTML(open(f"{OUT}/play_3d.html").read())

## Annotated video

In [ ]:
# Re-encode to H.264 so the browser can play it inline.
src = f"{OUT}/{os.path.splitext(os.path.basename(VIDEO))[0]}_annotated.mp4"
!ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -crf 26 /content/preview.mp4
from IPython.display import Video
Video("/content/preview.mp4", embed=True, width=960)